In [ ]:
import pandas as pd
import numpy as np
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import joblib
import random
import matplotlib.pyplot as plt
import os

## Data demo

In [ ]:
df1 = pd.read_csv("./data_labs/lab1.csv")
df1.tail()

## Data preparation

Load data and delete periods, when solar panels has not produced power for too long.

In [ ]:
H = 5 # [minutes]
H_HISTORY = 7 # [H_HISTORY * H] = [minutes]
H_FORECAST = 2
STRIDE_MIN = 7 # [minutes]
MAX_ZERO_LEN = 5 # [minutes]

PAC_COL = "Pac_combined"
TIME_COL = "created_at"
FEATURE_COLS = [
    "Pac_combined", "GPOA", "Tcell", "Tm", "Ta", "Ws", "f", "phi", "Q"
]


In [ ]:
def load_all_data(path_pattern="data_labs/*.csv"):
    dfs = []
    for fname in glob.glob(path_pattern):
        df = pd.read_csv(fname, parse_dates=[TIME_COL])
        df = df.sort_values(TIME_COL).reset_index(drop=True)
        dfs.append(df)
    return dfs

dfs = load_all_data()


In [ ]:
def drop_long_zero_and_segment(
        df,
        pac_col=PAC_COL,
        max_zero_len=MAX_ZERO_LEN,
        time_col=TIME_COL
):
    df = df.sort_values(time_col).reset_index(drop=True).copy()

    # time step
    diffs = df[time_col].diff().dropna()
    base_step = diffs.mode()[0] # time delta

    # drop time when there was no current
    df["_zero"] = (df[pac_col] == 0).astype(int)
    df["_grp"]  = (df["_zero"].diff() != 0).cumsum()

    long_zero_ids = []
    for gid, grp in df.groupby("_grp"):
        if grp["_zero"].iloc[0] == 1 and len(grp) > max_zero_len:
            long_zero_ids.append(gid)

    drop_mask = df["_grp"].isin(long_zero_ids)
    df_clean = df[~drop_mask].copy()
    df_clean.drop(columns=["_zero", "_grp"], inplace=True)

    if df_clean.empty:
        return [], base_step

    # split
    df_clean = df_clean.sort_values(time_col).reset_index(drop=True)
    diffs2 = df_clean[time_col].diff()
    diffs2.iloc[0] = base_step
    breaks = diffs2 > base_step * 1.5
    seg_ids = breaks.cumsum()

    segments = []
    for _, seg in df_clean.groupby(seg_ids):
        segments.append(seg.reset_index(drop=True))

    return segments, base_step




Create windows and split them on train and val.

In [ ]:
all_segments = []
base_step = None

for df in dfs:
    segs, step = drop_long_zero_and_segment(df)
    if base_step is None:
        base_step = step
    all_segments.extend(segs)

print(f"Segments num: {len(all_segments)}")
print(f"Time step: {base_step}")


In [ ]:
step_sec = base_step.total_seconds()

history_len  = int(H_HISTORY * H * 60 / step_sec) # 7H history
forecast_len = int(H_FORECAST * H * 60 / step_sec) # 2H target
stride_steps = int(STRIDE_MIN * 60 / step_sec)

total_len = history_len + forecast_len

print(
    "history_len =", history_len,
    "forecast_len =", forecast_len,
    "stride_steps =", stride_steps
)


In [ ]:
def build_windows(
        segments,
        history_len,
        forecast_len,
        stride_steps,
        time_col=TIME_COL
):
    total_len = history_len + forecast_len
    windows = []

    for seg_id, seg in enumerate(segments):
        if len(seg) < total_len:
            continue

        times = seg[time_col]

        for start in range(0, len(seg) - total_len + 1, stride_steps):
            sub_times = times.iloc[start : start + total_len]

            diffs = sub_times.diff().dropna()
            if diffs.empty:
                continue
            if not (diffs == diffs.iloc[0]).all():
                continue

            windows.append({
                "seg_id": seg_id,
                "start":  start,
                "t_start": sub_times.iloc[0],
                "t_end":   sub_times.iloc[-1],
            })

    return windows

windows = build_windows(all_segments, history_len, forecast_len, stride_steps)
print("Windows num:", len(windows))


In [ ]:
def split_train_val_windows(windows, val_fraction=0.1):
    windows_sorted = sorted(windows, key=lambda w: w["t_start"])
    n = len(windows_sorted)
    n_val = int(round(n * val_fraction))
    n_train = n - n_val
    train_w = windows_sorted[:n_train]
    val_w   = windows_sorted[n_train:]
    return train_w, val_w

train_windows, val_windows = split_train_val_windows(windows, val_fraction=0.1)

print("train:", len(train_windows))
print("val:", len(val_windows))


Normalize data.

In [ ]:
def collect_rows_from_windows(segments, windows, feature_cols):
    dfs = []
    for w in windows:
        seg = segments[w["seg_id"]]
        start = w["start"]
        end = start + history_len + forecast_len
        dfs.append(seg.iloc[start:end][feature_cols])
    return pd.concat(dfs).reset_index(drop=True)

train_df_for_scaler = collect_rows_from_windows(all_segments, train_windows, FEATURE_COLS)

In [ ]:
scaler = StandardScaler()
scaler.fit(train_df_for_scaler)

In [ ]:
joblib.dump(scaler, "models/scaler.pkl")

In [ ]:
normalized_segments = []

for seg in all_segments:
    seg_norm = seg.copy()
    seg_norm[FEATURE_COLS] = scaler.transform(seg[FEATURE_COLS])
    normalized_segments.append(seg_norm)

Prepare Dataset and Dataloader.

In [ ]:
class SolarDataset(Dataset):
    def __init__(
            self, segments, windows,
            feature_cols=FEATURE_COLS,
            history_len=history_len,
            forecast_len=forecast_len,
            pac_col=PAC_COL
    ):
        self.segments = segments
        self.windows = windows
        self.feature_cols = feature_cols
        self.history_len = history_len
        self.forecast_len = forecast_len
        self.pac_col = pac_col

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        seg = self.segments[w["seg_id"]]
        start = w["start"]
        H = self.history_len
        F = self.forecast_len

        x_df = seg.iloc[start : start + H]
        y_df = seg.iloc[start + H : start + H + F]

        x = x_df[self.feature_cols].values
        y = y_df[self.pac_col].values

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
        )


In [ ]:
train_ds = SolarDataset(normalized_segments, train_windows)
val_ds = SolarDataset(normalized_segments, val_windows)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds,   batch_size=256, shuffle=False)


## Baseline models

In [ ]:
class LSTMAttentionHybrid(nn.Module):
    def __init__(
            self,
            num_features, # F
            history_len, # L
            forecast_len, # P
            hidden_lstm=256,
            lstm_num_layers=4,
            lstm_dropout=0.25,
            hidden_mha=256,
            heads=4,
            mha_dropout=0.25,
            hidden_dense=128
    ):
        super().__init__()

        self.history_len = history_len # L
        self.forecast_len = forecast_len # P
        self.num_features = num_features # F

        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_lstm,
            num_layers=lstm_num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )

        if hidden_lstm != hidden_mha:
            self.proj = nn.Linear(hidden_lstm, hidden_mha)
        else:
            self.proj = nn.Identity()

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_mha,
            num_heads=heads,
            dropout=mha_dropout,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_mha, hidden_dense),
            nn.ReLU(),
            nn.Linear(hidden_dense, forecast_len) # PAC only
        )

    def forward(self, x):
        # x: [B, L, F]
        lstm_out, _ = self.lstm(x) # [B, L, hidden_lstm]

        lstm_out = self.proj(lstm_out) # [B, L, hidden_mha]

        attn_out, _ = self.attn(
            lstm_out, lstm_out, lstm_out
        ) # [B, H, hidden_mha]

        pooled = attn_out.mean(dim=1) # [B, hidden_mha]

        forecast = self.fc(pooled) # [B, P]

        return forecast


In [ ]:
class GatedResidual(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.proj = nn.Linear(input_dim, output_dim)
        self.gate = nn.Linear(output_dim, output_dim)

    def forward(self, x):
        h = self.proj(x)
        g = torch.sigmoid(self.gate(h))
        return g * torch.tanh(h) + (1 - g) * x


class VariableSelection(nn.Module):
    def __init__(self, num_features, hidden_dim):
        super().__init__()
        self.num_features = num_features
        self.weight_net = nn.Linear(num_features, num_features)
        self.proj = nn.Linear(num_features, hidden_dim)

    def forward(self, x):
        # x: [B, L, F]
        w = torch.softmax(self.weight_net(x.mean(dim=1)), dim=-1)  # [B, F]
        w = w.unsqueeze(1) # [B, 1, F]
        x_weighted = x * w # [B, L, F]
        return self.proj(x_weighted) # [B, L, H]


class TemporalFusionTransformer(nn.Module):
    def __init__(
        self,
        history_len,
        forecast_len,
        num_features,
        hidden_dim=128,
        lstm_dim=128,
        num_heads=4
    ):
        super().__init__()

        self.history_len = history_len
        self.forecast_len = forecast_len
        self.num_features = num_features

        # Variable selection network (feature importnce)
        self.vsn = VariableSelection(num_features, hidden_dim)

        # LSTM encoder (history)
        self.encoder = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=lstm_dim,
            batch_first=True
        )

        # Attention
        self.attn = nn.MultiheadAttention(
            embed_dim=lstm_dim,
            num_heads=num_heads,
            batch_first=True
        )

        # Decoder for P steps
        self.decoder = nn.Sequential(
            nn.Linear(lstm_dim, lstm_dim),
            nn.ReLU(),
            nn.Linear(lstm_dim, forecast_len)
        )

        # Gated skip connection
        self.gate = GatedResidual(lstm_dim, lstm_dim)

    def forward(self, x):
        B, L, F = x.size()
        x = self.vsn(x)
        enc_out, _ = self.encoder(x)
        attn_out, _ = self.attn(enc_out, enc_out, enc_out)
        fused = self.gate(attn_out)
        last = fused[:, -1, :]
        y_hat = self.decoder(last)
        return y_hat


In [ ]:
from tsai.models.PatchTST import PatchTST


class PatchTSTWrapper(nn.Module):
    def __init__(self, history_len, forecast_len, num_features, pac_channel_idx: int):
        super().__init__()
        self.pac_channel_idx = pac_channel_idx
        self.core = PatchTST(
            c_in=num_features,
            c_out=1,
            seq_len=history_len,
            pred_dim=forecast_len,
            n_layers=3,
            n_heads=16,
            d_model=128,
            d_ff=256,
            dropout=0.1,
            attn_dropout=0.0,
        )

    def forward(self, x):
        # x: [B, L, F] -> [B, F, L]
        x = x.permute(0, 2, 1)
        out = self.core(x) # [B, F, P]
        out = out[:, self.pac_channel_idx, :] # [B, P]
        return out


class ResCNNWrapper(nn.Module):
    def __init__(self, history_len, forecast_len, num_features):
        super().__init__()
        self.core = ResCNN(
            c_in=num_features,
            c_out=forecast_len,
            # опции можно оставить дефолтными
            coord=False,
            separable=False,
            zero_norm=False,
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)      # [B, F, L]
        out = self.core(x)          # [B, forecast_len]
        return out



## Training

In [ ]:
def train_model(
        model,
        train_loader,
        val_loader,
        epochs,
        lr,
        device,
        save_path="best_model.pth",
        min_delta=0.0,
        patience=10
):
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )

    best_val_loss = float("inf")
    best_epoch = 0
    no_improve = 0

    for epoch in range(1, epochs + 1):
        # train
        model.train()
        train_losses = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            pred = model(x)
            loss = F.mse_loss(pred, y)
            loss.backward()
            optimizer.step()

            train_losses += loss.item()

        train_loss = train_losses / len(train_loader)

        # val
        model.eval()
        val_losses = 0.0

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)

                pred = model(x)
                if pred.ndim == 3:
                    if pred.size(-1) == 1:
                        pred = pred.squeeze(-1)
                    else:
                        pred = pred[..., 0]

                loss = F.mse_loss(pred, y)

                val_losses += loss.item()

        val_loss = val_losses / len(val_loader)

        # early stopping
        improved = (best_val_loss - val_loss) > min_delta

        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improve = 0

            if save_path is not None:
                torch.save(model.state_dict(), save_path)  # save best weights
        else:
            no_improve += 1


        scheduler.step()

        print(
            f"[Epoch {epoch:03d}] "
            f"train={train_loss:.6f}, val={val_loss:.6f} | "
            f"best={best_val_loss:.6f}, (epoch {best_epoch})"
        )

        if no_improve >= patience:
            print(f"Early stopping triggered. No improvement for {patience} epochs.")
            break

    if save_path is not None:
        model.load_state_dict(torch.load(save_path, map_location=device))

    print(f"\nTraining complete. Best val_loss={best_val_loss:.6f} at epoch {best_epoch}.")
    return best_val_loss, model


In [ ]:
# model = LSTMAttentionHybrid(
#     num_features=9,
#     history_len=history_len,
#     forecast_len=forecast_len
# )

# model = TemporalFusionTransformer(
#     history_len=history_len,
#     forecast_len=forecast_len,
#     num_features=9,
# )

model = PatchTSTWrapper(
    history_len=history_len,
    forecast_len=forecast_len,
    num_features=9,
    pac_channel_idx=0,
)




In [ ]:
best_val, best_model = train_model(
    model,
    train_loader,
    val_loader,
    epochs=25,
    lr=1e-4,
    device="cuda",
    save_path=None
)


## Demo

In [ ]:
NUM_PLOTS = 2
history_len = model.history_len
forecast_len = model.forecast_len
F = 9
device = "cuda"

def denorm_pac(scaler, pac_vector, num_features):
    P = len(pac_vector)
    dummy = torch.zeros(P, num_features)
    dummy[:, 0] = torch.tensor(pac_vector)
    return scaler.inverse_transform(dummy.numpy())[:, 0]

indices = random.sample(range(len(val_ds)), NUM_PLOTS)

model.eval()

for i, idx in enumerate(indices):
    x, y_true = val_ds[idx]
    x = x.unsqueeze(0).to(device)
    y_true = y_true.unsqueeze(0).to(device)

    with torch.no_grad():
        y_pred = model(x)

    y_true_np = y_true.cpu().numpy().flatten()
    y_pred_np = y_pred.cpu().numpy().flatten()

    gt_full = denorm_pac(scaler, y_true_np, num_features=F)
    pred_full = denorm_pac(scaler, y_pred_np, num_features=F)

    plt.figure(figsize=(10, 5))
    plt.plot(gt_full, label="Ground Truth (PAC)", linewidth=2)
    plt.plot(pred_full, label="Predicted (PAC)", linewidth=2)
    plt.title(f"Validation Sample #{idx}")
    plt.xlabel("Forecast step (minutes)")
    plt.ylabel("PAC (denormalized)")
    plt.grid(True)
    plt.legend()
    plt.show()
